# I1: FP16 baseline, zero-change control, difficulty labels, head statistics

Builds the 300-problem stratified subset (commit `configs/gsm8k_300_stratified.json` afterwards), runs the bit-exact control, caches fp16 greedy generations with top-16 logprobs, labels difficulty by pass@4, records per-head attention entropy, fits the latency model. ~3 T4-hours. Save the output as a Kaggle dataset; every later notebook imports it.

Repo: https://github.com/Arubad/cot-under-compression.git  
Kaggle: Accelerator GPU T4 x2, Internet On. Results land in `/kaggle/working/results`.

In [ ]:
# --- setup (identical in every notebook) -------------------------------------------------
import os, subprocess, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
os.chdir(WORK)
if not os.path.exists('cot-under-compression'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Arubad/cot-under-compression.git'], check=True)
os.chdir('cot-under-compression')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
os.environ['RESULTS_ROOT'] = os.path.join(WORK, 'results')
# To reuse caches from an earlier notebook: Add Input -> that notebook's output, then set:
PREV = ''   # e.g. '/kaggle/input/cot-i1-baseline/results'
if PREV:
    os.environ['PREV_RESULTS'] = PREV
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


In [ ]:
!python -m src.run make_subset --config configs/baseline.yaml

In [ ]:
!python -m src.run baseline --config configs/baseline.yaml

In [ ]:
import pandas as pd, json
print(json.load(open(os.path.join(os.environ['RESULTS_ROOT'],'cache','control.json'))))
print(pd.read_csv(os.path.join(os.environ['RESULTS_ROOT'],'cache','head_stats.csv')).describe())
print(pd.read_parquet(os.path.join(os.environ['RESULTS_ROOT'],'cache','difficulty_gsm8k.parquet'))['stratum'].value_counts())

In [ ]:
# keep the results as this notebook's output (attach as a dataset to the next notebook)
!du -sh $RESULTS_ROOT && ls $RESULTS_ROOT